# CircuitGraft — Cloud RESCUE run (S2 drift + S3 audit + S4 discovery)

**Purpose:** produce the drift checkpoint, audit and both circuits on a Kaggle
GPU and push every artifact to the private HF Hub as soon as it exists, so a
session death can never lose completed work.

## HOW TO RUN (important — this is what went wrong before)

1. Right sidebar: **Accelerator → GPU T4 x2** (or P100), **Internet → ON**.
2. **Add-ons → Secrets**: `HF_TOKEN` attached to this notebook.
3. **CRITICAL: click "Save & Run All (Commit)"** — NOT "Run" on cells.
   Interactive sessions die and lose hours; a commit run is headless (you can
   close the browser), runs to completion, and saves all output + logs to the
   notebook version automatically. Enable notifications to get pinged when it
   finishes.
4. Expected wall time: **~6-9 h** (S2 ~2.5-3.5 h on T4, audit ~0.5 h,
   discovery ~2-4 h, uploads). Fits the Kaggle GPU session cap with margin.

Local machine keeps its own S2 running meanwhile — whichever finishes first
wins; artifacts are never mixed.


In [ ]:
# 1. Environment: keep the big HF cache OFF /kaggle/working (20 GB output cap)
import os

os.environ['HF_HOME'] = '/root/.cache/huggingface'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['ABSL_LOGGING_MIN_SEVERITY'] = '3'
os.environ['PYTHONUNBUFFERED'] = '1'

!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv
!free -g | head -2
!df -h /kaggle/working | tail -1


In [ ]:
# 2. Fresh code: clone + force tracked files to origin/main, then install
import os

%cd /kaggle/working
if not os.path.exists('/kaggle/working/circuit-graft'):
    !git clone --depth 1 https://github.com/0dvsh0/circuit-graft.git /kaggle/working/circuit-graft

%cd /kaggle/working/circuit-graft
!git fetch origin main
!git reset --hard origin/main
!git log --oneline -1

!pip uninstall -y torchao cudf-cu13 bigframes > /dev/null 2>&1 || true
!pip install --upgrade pip > /dev/null
!pip install --no-warn-conflicts safetune==0.1.12 circuitkit==0.2.0 auditkit==1.1.3 "curatorkit[generation,hf]==0.1.22" > /dev/null
!pip install --no-warn-conflicts "transformers==5.22.1" "bitsandbytes==0.50.2" "peft==0.21.2" "trl==1.14.2" "transformer-lens==3.8.0" "datasets==5.1.0" "accelerate==1.15.0" "huggingface-hub==1.33.0" sentence_transformers litellm python-dotenv typer rich > /dev/null
!pip uninstall -y torchao > /dev/null 2>&1 || true
!pip install -e . --no-deps > /dev/null
import importlib.util
assert importlib.util.find_spec('torchao') is None, 'torchao still installed'
import peft, transformers, torch
print('torch', torch.__version__, '| transformers', transformers.__version__, '| peft', peft.__version__)
print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')


In [ ]:
# 3. Auth: Kaggle secrets ONLY (no prompts — this is a headless commit run)
import os

try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret('HF_TOKEN')
except Exception as exc:
    raise RuntimeError(
        'HF_TOKEN not found. Add-ons > Secrets > add HF_TOKEN (attach to this '
        'notebook), then Save & Run All again.'
    ) from exc

os.environ['HF_TOKEN'] = hf_token
os.environ['HF_ORG'] = 'dubeydevansh'

from huggingface_hub import HfApi
api = HfApi(token=hf_token)
try:
    files = api.list_repo_files('CohereLabs/tiny-aya-global', repo_type='model')
    print('gated model access OK,', len(files), 'files')
except Exception as exc:
    raise RuntimeError(
        'tiny-aya-global is BLOCKED for this token. Accept the licence on the '
        'HF website with the SAME account that owns HF_TOKEN, then re-run.'
    ) from exc


In [ ]:
# 4. Data: import the verified snapshot (no generation, no human-review step)
import json
import os
from pathlib import Path

from huggingface_hub import snapshot_download

DATA_REPO = os.environ.get('CG_DATA_REPO', 'dubeydevansh/circuitgraft-phase4-data')
summary_path = Path('artifacts/data/summary.json')
if not summary_path.exists():
    snapshot_download(repo_id=DATA_REPO, repo_type='dataset', local_dir='artifacts/data',
                      token=os.environ['HF_TOKEN'])
summary = json.loads(summary_path.read_text(encoding='utf-8'))
print('data run_id:', summary.get('run_id'), '| G1:', summary.get('gate_g1', {}).get('status'))
assert Path('artifacts/data/medmcqa_sft/train.jsonl').exists(), 'SFT rows missing'
assert Path('artifacts/data/pairs/safety.jsonl').exists(), 'pairs missing'


In [ ]:
# 5. S2 drift — batch 1 x grad_accum 32 (validated OOM-free shape)
import subprocess, sys, os

r = subprocess.run(
    [sys.executable, '-m', 'circuitgraft.cli', 'drift', '--config', 'configs/drift_t4.yaml'],
    env={**os.environ, 'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True'},
)
if r.returncode != 0:
    raise RuntimeError(f'drift failed with return code {r.returncode} — checkpoint saves every 100 steps')


In [ ]:
# 6. Verify drift outputs before anything else can fail
import json
from pathlib import Path

d = Path('artifacts/drift/scenarioA_seed0')
files = sorted(p for p in d.rglob('*') if p.is_file())
total_gb = sum(p.stat().st_size for p in files) / 2**30
print(f'{len(files)} files, {total_gb:.2f} GB')
for p in files:
    print(f'  {p.relative_to(d)}  {p.stat().st_size / 2**20:.1f} MB')
merged = [p for p in files if p.suffix == '.safetensors' and 'merged' in p.name]
assert merged, 'no merged checkpoint found'
print('merged checkpoint:', merged[0].name)


In [ ]:
# 7. UPLOAD the drift artifacts NOW — before any stage that could fail
import os

from huggingface_hub import HfApi

CLOUD_REPO = 'dubeydevansh/circuitgraft-phase4-cloud'
api = HfApi(token=os.environ['HF_TOKEN'])
api.create_repo(CLOUD_REPO, repo_type='dataset', private=True, exist_ok=True)
api.upload_folder(folder_path='artifacts/drift', repo_id=CLOUD_REPO, repo_type='dataset',
                  path_in_repo='artifacts/drift', commit_message='rescue: S2 drift complete')
for p in Path('results').rglob('*.json'):
    api.upload_file(path_or_fileobj=str(p), path_in_repo=str(p).replace('\\', '/'),
                    repo_id=CLOUD_REPO, repo_type='dataset',
                    commit_message='rescue: results json')
print('drift artifacts uploaded -> https://huggingface.co/datasets/' + CLOUD_REPO)


In [ ]:
# 8. S3 audit (aligned vs drifted)
import subprocess, sys, os

r = subprocess.run([sys.executable, '-m', 'circuitgraft.cli', 'audit',
                    '--config', 'configs/eval.yaml', '--models', 'aligned,drifted'])
print('audit return code:', r.returncode)


In [ ]:
# 9. S4 discovery — both circuits (safety on aligned, domain on drifted),
# ig_steps 2, cross_check disabled (logged prereg deviations). Runs AFTER the
# S2 upload; failures here cannot undo the drift artifacts on the Hub.
import subprocess, sys, os

r = subprocess.run(
    [sys.executable, '-m', 'circuitgraft.cli', 'discover', '--config', 'configs/discover.yaml'],
    env={**os.environ, 'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True'},
)
print('discover return code:', r.returncode)


In [ ]:
# 10. Final upload: circuits + audit + any remaining artifacts/results
import os
from pathlib import Path

from huggingface_hub import HfApi

CLOUD_REPO = 'dubeydevansh/circuitgraft-phase4-cloud'
api = HfApi(token=os.environ['HF_TOKEN'])
if Path('artifacts/discover').exists():
    api.upload_folder(folder_path='artifacts/discover', repo_id=CLOUD_REPO,
                      repo_type='dataset', path_in_repo='artifacts/discover',
                      commit_message='rescue: S4 circuits complete')
api.upload_folder(folder_path='results', repo_id=CLOUD_REPO, repo_type='dataset',
                  path_in_repo='results', commit_message='rescue: results final')
print('ALL cloud artifacts -> https://huggingface.co/datasets/' + CLOUD_REPO)
print()
print('Next on the local machine (after this run finishes):')
print('  git pull')
print('  python scripts/import_cloud_artifacts.py')


## After the run finishes

On the **local machine**:
```bash
git pull
python scripts/import_cloud_artifacts.py   # pulls drift + circuits + audit from the Hub
```
Then continue with S5 baselines → S6 graft → selection/final evals → attacks → report → publish.
